### 1. Merge & Pre-processing

### Introduction and Goals
The cleaned transactions (`train`, `test`) and the cleaned customer table (`clientes`) are merged, and the merged table is pre-processed in order: Missing Values, Outliers, Encoding and Scaling. The fill plans of section 1.6 are scored on **temporal** validation splits (train on the past, validate on the period after it), and every step that learns from the data is fitted on the training rows only.

| Section | Action | Output file (train / test) |
| :--- | :--- | :--- |
| **Loading Data** | Load `train`, `clientes` and `test` | `(no file written)` |
| **1.1** | Merge transactions with customers (left join, many-to-one) | `merge_1_2.csv` / `merge_1_2_test.csv` |
| **1.2** | Separate identifiers, target, numeric and categorical features | `(no file written)` |
| **1.3** | Missing values overview | `(no file written)` |
| **1.4** | Evaluation setup: the model matrix, the fill and the scorer that every comparison uses | `(no file written)` |
| **1.5** | Temporal splits and the baseline that every fill plan has to beat | `(no file written)` |
| **1.6** | Compare fill plans (1.6.1, 1.6.2) and fill the missing values with `median` / `mode` (1.6.3) | `merge_1_6.csv` / `merge_1_6_test.csv` |
| **1.7** | Outlier detection (1.7.1) and `log1p` treatment (1.7.2) | `merge_1_7.csv` / `merge_1_7_test.csv` |
| **1.8** | Rare-level grouping and one-hot encoding (1.8.1); `StandardScaler` (1.8.2) | `1_8_1.csv` / `1_8_1_test.csv`, `1_8_2.csv` / `1_8_2_test.csv` |

**Conventions:** in 1.6 the fill plans are compared on average precision (AP), paired on the same splits; an alternative is only adopted when it beats the reference by more than two standard errors, otherwise the simplest option stays. F1 is shown beside AP but does not decide. Sections 1.7 and 1.8 apply a fixed recipe (`log1p`, one-hot, `StandardScaler`) without scoring alternatives.

### Imports

In [1]:
import warnings
from collections import namedtuple
from functools import partial

import numpy as np
import pandas as pd

from sklearn.exceptions import ConvergenceWarning
from sklearn.experimental import enable_iterative_imputer  # noqa: F401 (needed for IterativeImputer)
from sklearn.impute import IterativeImputer, KNNImputer, SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, f1_score
from sklearn.model_selection import TimeSeriesSplit
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# Same seed everywhere, so every run gives the same splits and draws
RANDOM_STATE = 42

____________________________________________________________________________________________________

### Loading Data

In [2]:
trainFile = "../../data/trainDataset/train_2_5_1.csv"
clientFile = "../../data/clientesDataset/clientes_clean.csv"
testFile = "../../data/testDataset/test_2_5_1.csv"

train = pd.read_csv(trainFile)
clientes = pd.read_csv(clientFile)
test = pd.read_csv(testFile)

____________________________________________________________________________________________________

## 1.1. Merging the Datasets

In [3]:
train_clientes = train.merge(clientes, on="customer_id", how="left", validate="many_to_one")
assert len(train_clientes) == len(train)

test_clients = test.merge(clientes, on="customer_id", how="left", validate="many_to_one")
assert len(test_clients) == len(test)

display(train_clientes.head())
display(test_clients.head())

,Total a Pagar,Total Bruto,Total Liquido,Total Imp_,Total Portes,Total Desc_ Global,Total Desc_ Linha,Custo,N_ Detalhes,invoice_count_current_purchase,...,island,municipality,city,postal_area,postal_code,street,address,company_name,customer_since_year,since_2015_or_earlier
0,558.11,510.90,453.75,104.36,0.0,0.0,57.15,153.099911,4.0,1.0,...,(mainland),Municipality MU89: Council of Last Minute Edits,City CY286: Cached Result Falls,Postal area PA228: Hidden Sheet Sector,Postal code PC444: #REF! Delivery Route,Street ST303: Absolute Reference Terrace,Address AD761: Cell Without a Home,Excel Is Not A Database Ltd 1013,2024,0
1,978.80,830.02,795.77,183.03,0.0,0.0,34.25,472.943134,32.0,1.0,...,Island IS8: The Pivot Archipelago,Municipality MU11: Council of Last Minute Edits,City CY147: VLOOKUP Bay,Postal area PA260: Hidden Sheet Sector,Postal code PC636: #REF! Delivery Route,NaN,Address AD382: Cell Without a Home,Excel Is Not A Database Ltd 35,2015,1
2,10.33,12.00,8.40,1.93,0.0,0.0,3.60,6.097485,1.0,1.0,...,(mainland),Municipality MU70: Council of Last Minute Edits,City CY265: Where-less Query Falls,Postal area PA84: Hidden Sheet Sector,Postal code PC200: #REF! Delivery Route,NaN,Address AD723: Cell Without a Home,Excel Is Not A Database Ltd 826,2017,0
3,856.83,696.61,696.61,160.22,0.0,0.0,0.00,408.167275,9.0,1.0,...,Island IS7: The Pivot Archipelago,Municipality MU12: Council of Last Minute Edits,NaN,Postal area PA253: Hidden Sheet Sector,Postal code PC558: #REF! Delivery Route,Street ST144: Last Save Court,Address AD399: Cell Without a Home,Excel Is Not A Database Ltd 530,2018,0
4,1616.10,1488.85,1313.90,302.20,0.0,0.0,174.95,485.401864,33.0,1.0,...,(mainland),Municipality MU142: Council of Last Minute Edits,City CY359: Spill Error Ridge,Postal area PA136: Hidden Sheet Sector,Postal code PC300: #REF! Delivery Route,Street ST395: AutoFill Path,Address AD928: Cell Without a Home,Excel Is Not A Database Ltd 220,2017,0


,Total a Pagar,Total Bruto,Total Liquido,Total Imp_,Total Portes,Total Desc_ Global,Total Desc_ Linha,Custo,N_ Detalhes,invoice_count_current_purchase,...,island,municipality,city,postal_area,postal_code,street,address,company_name,customer_since_year,since_2015_or_earlier
0,286.37,582.05,232.82,53.55,0.0,0.0,349.23,104.409267,22,1,...,(mainland),Municipality MU70: Council of Last Minute Edits,NaN,Postal area PA84: Hidden Sheet Sector,Postal code PC204: #REF! Delivery Route,NaN,Address AD724: Cell Without a Home,Excel Is Not A Database Ltd 32,2015,1
1,1949.25,1725.00,1725.00,224.25,0.0,0.0,0.00,0.000000,1,1,...,Island IS5: The Pivot Archipelago,Municipality MU17: Council of Last Minute Edits,City CY194: PowerPoint New Hills,Postal area PA252: Hidden Sheet Sector,Postal code PC540: #REF! Delivery Route,NaN,Address AD523: Cell Without a Home,Excel Is Not A Database Ltd 450,2015,1
2,2738.71,4114.83,2738.71,0.00,0.0,0.0,1376.12,1019.466586,99,1,...,(foreign),(foreign),NaN,(foreign),Postal code PC29: #REF! Delivery Route,NaN,Address AD26: Cell Without a Home,Excel Is Not A Database Ltd 402,2021,0
3,726.98,591.04,591.04,135.94,0.0,0.0,0.00,341.649008,39,2,...,Island IS7: The Pivot Archipelago,Municipality MU9: Council of Last Minute Edits,City CY133: Conditional Format Crossing,Postal area PA249: Hidden Sheet Sector,Postal code PC518: #REF! Delivery Route,Street ST116: Filtered View Crescent,Address AD338: Cell Without a Home,Excel Is Not A Database Ltd 460,2024,0
4,670.76,555.49,545.33,125.43,0.0,0.0,10.16,326.715748,13,1,...,Island IS7: The Pivot Archipelago,Municipality MU12: Council of Last Minute Edits,City CY157: Conditional Format Bay,Postal area PA253: Hidden Sheet Sector,Postal code PC555: #REF! Delivery Route,NaN,Address AD397: Cell Without a Home,Excel Is Not A Database Ltd 398,2025,0


In [4]:
outputFile = "../../data/preprocessing/merge_1_2.csv"
outputFileTest = "../../data/preprocessing/merge_1_2_test.csv"

train_clientes.to_csv(outputFile, index=False)
test_clients.to_csv(outputFileTest, index=False)

**Conclusions:**
* **Lossless merge:** the merge is a left join with `validate="many_to_one"` (one row per `customer_id` in `clientes`) and the `assert` on the number of rows passes, so no transaction was lost or duplicated. The merged tables have **57 columns** in train and **56** in test (the test has no target).
* **Customer coverage:** the join only brings in the customers that appear in the transactions; the remaining customers of `clientes` are simply not used.
* **New missing values:** the customer attributes bring their own gaps into the merged table, mostly in identifier-like columns (`street`, `city`, `postal_code`); the descriptive ones are almost complete. They are quantified in section 1.3 and handled together with the `train` gaps in section 1.6.

____________________________________________________________________________________________________

## 1.2. Feature Type Separation
Imputation and encoding work differently on numbers and on labels (a median or a KNN fill needs numbers, a mode or an `"Unknown"` level works on labels), so the merged columns are split into groups first:

1. **Target:** `repeat_purchase_90d`, the variable to predict.
2. **Identifiers:** `ID` (one per row) and `customer_id` (the join key with the customer table).
3. **Not to be modelled:** `company_name`, `address`, `street` and `postal_code` are distinct for almost every customer, so they name a customer instead of describing one, and `random_noise` carries no information by construction. They are not removed here (see the conclusions below): the four text columns are dropped in 1.8.1, and the feature selection notebook leaves all five out.
4. **Numeric** and **categorical** features: everything else, separated by data type.

In [5]:
train_clients = pd.read_csv(outputFile)
test_clients = pd.read_csv(outputFileTest)

def separate_feature_types(
    dataframe,
    target_col="repeat_purchase_90d",
    identifiers=["ID", "customer_id"],
    dataset_label="Dataset",
):
  """Numeric and categorical feature columns of `dataframe` (identifiers and target left out).

  Prints the counts and returns the feature table, the target (None when the
  table has no target), and the numeric and categorical column lists.
  """
  excluded_cols = identifiers + (
      [target_col] if target_col in dataframe.columns else []
  )
  features_list = [c for c in dataframe.columns if c not in excluded_cols]

  numeric = [
      c for c in features_list if pd.api.types.is_numeric_dtype(dataframe[c])
  ]
  categorical = [c for c in features_list if c not in numeric]

  X_features = dataframe[numeric + categorical]
  y_target = dataframe[target_col] if target_col in dataframe.columns else None

  print(f"=== Feature Separation ({dataset_label}) ===")
  print(
      f"{len(numeric)} numeric | {len(categorical)} categorical |"
      f" {len(identifiers)} identifiers"
  )
  print(f"categorical: {categorical}\n")

  return X_features, y_target, numeric, categorical


# Separate the variables of the training and test tables
X_train, y_train, numeric_train, categorical_train = separate_feature_types(
    train_clients, dataset_label="Train"
)

X_test, _, numeric_test, categorical_test = separate_feature_types(
    test_clients, dataset_label="Test"
)


=== Feature Separation (Train) ===
38 numeric | 16 categorical | 2 identifiers
categorical: ['payment_type', 'salesperson', 'commercial_zone', 'transport_method', 'warehouse', 'document_series', 'country_or_region', 'commercial_region', 'island', 'municipality', 'city', 'postal_area', 'postal_code', 'street', 'address', 'company_name']

=== Feature Separation (Test) ===
38 numeric | 16 categorical | 2 identifiers
categorical: ['payment_type', 'salesperson', 'commercial_zone', 'transport_method', 'warehouse', 'document_series', 'country_or_region', 'commercial_region', 'island', 'municipality', 'city', 'postal_area', 'postal_code', 'street', 'address', 'company_name']



**Conclusions:**
* **54 features** come out of the separation: **38 numeric** and **16 categorical** (plus the two identifiers and the target). The target has no missing values and is fairly balanced (about 56% of purchases are repeated within 90 days). Train and test give the same counts.
* **The columns not to be modelled are still in these lists:** `company_name`, `address`, `street` and `postal_code` (categorical) and `random_noise` (numeric) are not removed here, so they take part in the fills of 1.6 and in the comparisons of 1.4 to 1.6. The four text columns are dropped in 1.8.1; `random_noise` is never dropped and is still standardised in 1.8.2.
* **Numbers that are labels:** some numeric columns are categories written as numbers: `purchase_month`, `purchase_dayofweek`, `purchase_quarter` and the 0/1 flags `is_first_purchase`, `is_weekend` and `since_2015_or_earlier`. None of them has missing values, so they do not affect the imputation. They are not encoded as categories later: they stay numeric and are standardised with the rest in 1.8.2.

____________________________________________________________________________________________________

## 1.3. Missing Values Overview

In this step, we measure the missing values, which gives us some information about which features still have gaps after the merge, how many, and whether a missing value says something about the target.

In [6]:
def generate_missing_overview(dataframe, target_col="repeat_purchase_90d", dataset_label="Dataset"):
    """Missing values per feature (identifiers and target left out).

    When the table has the target, it also gives the repeat rate of the rows
    where the value is missing and of the rows where it is present.
    """
    identifiers = ["ID", "customer_id"]
    excluded = identifiers + ([target_col] if target_col in dataframe.columns else [])
    
    feature_cols = [c for c in dataframe.columns if c not in excluded]
    
    numeric_cols = [c for c in feature_cols if pd.api.types.is_numeric_dtype(dataframe[c])]
    categorical_cols = [c for c in feature_cols if c not in numeric_cols]
    
    features_df = dataframe[feature_cols]
    
    missing = features_df.isna().sum()
    
    overview_dict = {
        "group": ["numeric" if c in numeric_cols else "categorical" for c in features_df.columns],
        "n_missing": missing,
        "pct_missing": (missing / len(features_df) * 100).round(1)
    }
    
    if target_col in dataframe.columns:
        target_series = dataframe[target_col]
        overview_dict["target_if_missing"] = [target_series[features_df[c].isna()].mean() for c in features_df.columns]
        overview_dict["target_if_present"] = [target_series[features_df[c].notna()].mean() for c in features_df.columns]
    
    overview_df = (
        pd.DataFrame(overview_dict)
        .query("n_missing > 0")
        .sort_values("n_missing", ascending=False)
        .round(3)
    )
    
    print(f"=== MISSING VALUES OVERVIEW ({dataset_label}) ===")
    print(f"rows with at least one missing value: {features_df.isna().any(axis=1).sum():,} of {len(features_df):,}\n")
    return overview_df

# Make sure the global variables 'features' and 'repeat_purchase' are available for the rest of the code (Train)
features = train_clientes[[c for c in train_clientes.columns if c not in ["ID", "customer_id", "repeat_purchase_90d"]]]
repeat_purchase = train_clientes["repeat_purchase_90d"]

# Executes for the Training dataset
overview_train = generate_missing_overview(train_clientes, dataset_label="Train")
display(overview_train)


=== MISSING VALUES OVERVIEW (Train) ===
rows with at least one missing value: 5,155 of 6,469



,group,n_missing,pct_missing,target_if_missing,target_if_present
street,categorical,3808,58.9,0.587,0.520
city,categorical,2128,32.9,0.553,0.563
postal_code,categorical,223,3.4,0.422,0.565
spend_365d,numeric,210,3.2,0.552,0.560
transport_method,categorical,196,3.0,0.551,0.560
payment_type,categorical,195,3.0,0.564,0.560
warehouse,categorical,195,3.0,0.636,0.557
salesperson,categorical,194,3.0,0.598,0.559
postal_area,categorical,65,1.0,0.492,0.560
commercial_region,categorical,35,0.5,0.343,0.561


**Conclusions:**
* **Most rows have a gap, but few features do:** 5,155 of the 6,469 training rows (about 80%) have at least one missing value, spread over 12 features, so dropping rows is not an option (compared in 1.6.3).
* **Almost all gaps are in categorical features.** The largest come from the customer table: `street` (58.9%), `city` (32.9%) and `postal_code` (3.4%). `transport_method`, `payment_type`, `warehouse` and `salesperson` have about 3% each, and `postal_area` (1.0%), `commercial_region` (0.5%), `island` and `municipality` (0.1%) are nearly complete. `spend_365d` (3.2%) is the **only numeric** feature with gaps, so the numeric fill changes a single column.
* **Does missing say something about the target?** For the large gaps, not much: the repeat rate is 0.553 against 0.563 for `city` and 0.552 against 0.560 for `spend_365d`. The larger differences sit in small groups (`warehouse` 0.636, `salesperson` 0.598, `postal_code` 0.422, `commercial_region` 0.343), and `island` and `municipality` show 0.000 on only 4 rows, so these rates are noisy and only suggestive. Section 1.6 tests how to fill the gaps.
* **Only the train table is profiled here:** the test table is filled with the training medians and modes in 1.6.3.

____________________________________________________________________________________________________

## 1.4. Evaluation Setup

Every comparison below runs through the same three tools, written once, as in the course practical: the **model matrix** (`design_matrix`), the **fill** that makes a split ready for a model (`fill_missing`), and the **scorer** (`held_out_scores`), which trains a logistic regression on the training rows of each split and scores it on the untouched validation rows.

The scorer returns two measures. **Average precision (AP)** decides every comparison: it reads the predicted probabilities and asks whether the customers who come back are ranked above the ones who do not, without choosing a cut-off. **F1**, the competition metric, is shown beside it; it depends on cutting the probabilities at 0.5, and a different cut-off gives a different F1, so it does not decide. The cut-off is chosen, and F1 reported, for the final model (model notebook).

In [7]:
# The table every comparison works on: the features plus the target
frame = features.join(repeat_purchase)

# Logistic regression may stop before converging on unscaled columns; scaling is only applied in 1.8.2
warnings.filterwarnings("ignore", category=ConvergenceWarning)


def design_matrix(frame, numeric, categorical):
    """Numbers as they are, everything else one-hot. Identifiers never enter."""
    encoder = OneHotEncoder(sparse_output=False, dtype=float)
    encoded = encoder.fit_transform(frame[categorical].astype("string"))
    text = pd.DataFrame(
        encoded,
        columns=encoder.get_feature_names_out(categorical),
        index=frame.index,
    )
    return pd.concat([frame[numeric], text], axis=1)


def fill_missing(train, test, plan):
    """Fill every missing value with imputers fitted on `train` alone.

    `plan` is a list of (columns, imputer) pairs: each imputer is fitted on the
    training rows of its own columns and fills those columns in both halves, so
    the validation rows never help compute a fill. Every gap becomes np.nan first,
    because scikit-learn does not read pd.NA as a gap; the KNN and iterative
    imputers work on standardised columns and return their fills in the columns'
    own units.
    """
    train, test = train.copy(), test.copy()
    for columns, imputer in plan:
        blocks = [half[columns].astype(object).fillna(np.nan).infer_objects()
                  for half in (train, test)]
        if isinstance(imputer, (KNNImputer, IterativeImputer)):
            scale = StandardScaler().fit(blocks[0])
            imputer.fit(scale.transform(blocks[0]))
            filled = [scale.inverse_transform(imputer.transform(scale.transform(block)))
                      for block in blocks]
        else:
            imputer.fit(blocks[0])
            filled = [imputer.transform(block) for block in blocks]
        train[columns], test[columns] = filled
    return train, test


# A model cannot fit a missing value, so every comparison fills inside its own split.
# Until section 1.6 chooses a plan, that fill is the training median for the numbers
# and the training mode for the labels.
median_mode = [(numeric_train, SimpleImputer(strategy="median")),
               (categorical_train, SimpleImputer(strategy="most_frequent"))]

Scored = namedtuple("Scored", "ap f1 iterations predictions")


def sem(values):
    """Standard error of the mean: how far it would move on new splits."""
    return values.std(ddof=1) / np.sqrt(len(values))


def held_out_scores(prepare, frame, splits, numeric, categorical,
                    target="repeat_purchase_90d", scaler=None):
    """Score one treatment on each split's untouched validation rows.

    `prepare(train, test)` applies the treatment, fitted on `train` only. The
    result holds, per split, the validation average precision (`.ap`), the F1
    at the 0.5 cut-off (`.f1`), the solver's iterations (`.iterations`) and the
    (true labels, predicted probabilities) pair (`.predictions`). `scaler`, when
    given, is fitted on the training rows and applied to both.
    """
    ap, f1, iterations, predictions = [], [], [], []
    for train_rows, test_rows in splits:
        train, test = prepare(frame.iloc[train_rows], frame.iloc[test_rows])
        matrix = design_matrix(pd.concat([train, test]), numeric, categorical)
        a, b = matrix.iloc[:len(train)], matrix.iloc[len(train):]
        if scaler is not None:
            fitted = scaler.fit(a)
            a, b = fitted.transform(a), fitted.transform(b)
        model = LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)
        model.fit(a, train[target])
        probabilities = model.predict_proba(b)[:, 1]
        ap.append(average_precision_score(test[target], probabilities))
        f1.append(f1_score(test[target], probabilities >= 0.5))
        iterations.append(int(model.n_iter_[0]))
        predictions.append((test[target].to_numpy(), probabilities))
    return Scored(np.array(ap), np.array(f1), np.array(iterations), predictions)

____________________________________________________________________________________________________

## 1.5. Temporal Splits
The course practical (`week_03_deepen_exploration`) scores every candidate on 20 **random** splits. That works for its data, which has no dates, but here a random split would leak the future:

* **The target looks ahead.** `repeat_purchase_90d` is 1 when the same customer buys again within 90 days, so the label of one purchase depends on the customer's *next* purchase. A random split can put that next purchase in training while it scores the earlier one, and the same customer's purchases on both sides look alike.
* **The test is later in time.** `test.csv` covers May 2025 to October 2026, after every purchase in `train`, and the repeat rate rises over the years. A random split hides that drift, and the competition rules penalise a large gap between the performance a notebook reports and its Kaggle result.

So every comparison in section 1.6, and the final model's reported performance, uses splits **in time**: each one trains on the past and validates on the period after it. The rest of the practical's method stays: every step that learns from the data is fitted on the training rows of each split only, rows are never dropped from the validation side, and every candidate is scored on the same validation rows.

* **Rolling origin:** the rows are ordered by purchase date and cut into 10 consecutive blocks with `TimeSeriesSplit`; each block is validated by a model trained only on the purchases before it.
* **Enough history:** the first blocks have very few purchases before them, so a block is kept only when its training side has at least 2,000 rows.
* **90-day gap:** the training purchases from the 90 days before each validation block are left out, because their labels depend on purchases inside the block. A similar gap separates `train` from `test.csv` (98 days).
* **Same scorer:** `held_out_scores` accepts any list of (training rows, validation rows) pairs, so these splits use it unchanged.

In [8]:
# Purchase date, rebuilt from its parts (train_2_5_1 no longer carries purchase_date)
purchase_dates = pd.to_datetime(frame[["purchase_year", "purchase_month", "purchase_day"]]
                                .set_axis(["year", "month", "day"], axis=1))

# Positions of the rows in date order; TimeSeriesSplit cuts this order into blocks
order = np.argsort(purchase_dates.to_numpy(), kind="stable")
GAP = pd.Timedelta(days=90)
MIN_TRAIN = 2000

time_splits = []
for train_block, validation_block in TimeSeriesSplit(n_splits=10).split(order):
    train_rows, validation_rows = order[train_block], order[validation_block]
    # Leave out the training purchases whose 90-day window reaches the validation block
    start = purchase_dates.iloc[validation_rows].min()
    train_rows = train_rows[(purchase_dates.iloc[train_rows] < start - GAP).to_numpy()]
    # The first blocks have too little history to train on
    if len(train_rows) >= MIN_TRAIN:
        time_splits.append((train_rows, validation_rows))

print(f"{len(time_splits)} temporal splits kept of 10")

7 temporal splits kept of 10


In [9]:
# Check: each split trains on the past only and validates on the period after it
pd.DataFrame([{"training until": purchase_dates.iloc[tr].max().date(),
               "validation from": purchase_dates.iloc[va].min().date(),
               "validation until": purchase_dates.iloc[va].max().date(),
               "training rows": len(tr), "validation rows": len(va),
               "training rate": repeat_purchase.iloc[tr].mean(),
               "validation rate": repeat_purchase.iloc[va].mean()}
              for tr, va in time_splits]).round(3)

,training until,validation from,validation until,training rows,validation rows,training rate,validation rate
0,2019-03-10,2019-06-09,2020-03-28,2229,588,0.563,0.563
1,2019-12-28,2020-03-28,2021-03-06,2785,588,0.553,0.604
2,2020-12-05,2021-03-06,2021-11-13,3399,588,0.562,0.537
3,2021-08-14,2021-11-13,2022-10-02,3898,588,0.562,0.502
4,2022-07-05,2022-10-04,2023-07-28,4524,588,0.558,0.502
5,2023-04-29,2023-07-29,2024-04-13,5131,588,0.548,0.622
6,2024-01-13,2024-04-13,2025-02-15,5665,588,0.554,0.605


In [10]:
# The baseline every later section has to beat, on the temporal splits
time_baseline = held_out_scores(partial(fill_missing, plan=median_mode),
                                frame, time_splits, numeric_train, categorical_train)

# Predicting a repeat purchase for every customer, the floor any model must clear.
# A constant score ranks nobody, so its AP is just the share of repeat purchases.
all_positive_f1 = np.array([f1_score(repeat_purchase.iloc[va], np.ones(len(va))) for _, va in time_splits])
all_positive_ap = np.array([repeat_purchase.iloc[va].mean() for _, va in time_splits])

print(f"temporal baseline    AP {time_baseline.ap.mean():.4f} +/- {sem(time_baseline.ap):.4f}"
      f"   F1 {time_baseline.f1.mean():.4f} +/- {sem(time_baseline.f1):.4f}")
print(f"predict-all-positive AP {all_positive_ap.mean():.4f}             F1 {all_positive_f1.mean():.4f}")
print(f"AP per split {np.round(time_baseline.ap, 4)}")

temporal baseline    AP 0.8325 +/- 0.0074   F1 0.7252 +/- 0.0069
predict-all-positive AP 0.5622             F1 0.7186
AP per split [0.8362 0.8234 0.8539 0.803  0.8145 0.8536 0.8428]


**Conclusions:**
* **7 of the 10 blocks are kept.** Their training sides run from 2,229 to 5,665 purchases, each validation block holds 588, and the validation periods cover June 2019 to February 2025, so every split trains on the past and is scored on the months after it.
* **The repeat rate drifts over time:** it ranges from 0.50 to 0.62 across the validation blocks, a movement the random splits would have averaged away.
* **The instrument ranks customers well:** the baseline (median/mode fill, logistic regression) reaches **AP = 0.8323 ± 0.0074**, against **0.5622** for a constant prediction, whose AP is simply the share of repeat purchases. Every treatment in section 1.6 is compared against this baseline, on these 7 splits.
* **Its F1 is only slightly above the trivial one:** at the 0.5 cut-off the baseline scores **0.7274 ± 0.0075**, against **0.7186** for predicting a repeat purchase for everyone. As about 56% of the purchases repeat, "predict everyone" is a strong floor for F1, and F1 depends on the cut-off, so AP is the measure that decides the comparisons. The cut-off is chosen for the final model.

____________________________________________________________________________________________________

## 1.6. Missing Values Comparison and Selection

In [11]:
class ProportionalImputer:
    """Fill each missing label with one drawn from its column's training labels,
    in their training shares.

    scikit-learn has no such imputer, and `fill_missing` needs only `fit` and
    `transform`, so a short class is enough.
    """

    def __init__(self, random_state=None):
        self.random_state = random_state

    def fit(self, frame):
        self.shares = {c: frame[c].value_counts(normalize=True) for c in frame}
        return self

    def transform(self, frame):
        draws = np.random.default_rng(self.random_state)
        frame = frame.copy()
        for column, shares in self.shares.items():
            gaps = frame[column].isna()
            if gaps.any():
                frame.loc[gaps, column] = draws.choice(
                    shares.index, size=gaps.sum(), p=shares.to_numpy())
        return frame



### 1.6.1. Fill Plans

In this subsection we define the possible imputers and the fill plans, and then score every plan with the `held_out_scores` function on the temporal splits of 1.5.

In [12]:
# One imputer per fill. "own level" gives a missing label a category of its
# own instead of the common one.
fills = {"median": SimpleImputer(strategy="median"),
         "mean": SimpleImputer(strategy="mean"),
         "zero": SimpleImputer(strategy="constant", fill_value=0),
         "mode": SimpleImputer(strategy="most_frequent"),
         "own level": SimpleImputer(strategy="constant", fill_value="(missing)"),
         "proportional": ProportionalImputer(random_state=RANDOM_STATE)}

plans = {
    "median / mode": [(numeric_train, fills["median"]), (categorical_train, fills["mode"])],
    "mean / mode": [(numeric_train, fills["mean"]), (categorical_train, fills["mode"])],
    "zero / mode": [(numeric_train, fills["zero"]), (categorical_train, fills["mode"])],
    "median / own level": [(numeric_train, fills["median"]), (categorical_train, fills["own level"])],
    "median / proportional": [(numeric_train, fills["median"]), (categorical_train, fills["proportional"])],
}

plan_scores = {name: held_out_scores(partial(fill_missing, plan=plan),
                                 frame, time_splits, numeric_train, categorical_train)
               for name, plan in plans.items()}


def fill_with_indicator(train, test, plan):
    """Mark where spend_365d was missing in its own 0/1 column, then fill."""
    train, test = (f.assign(spend_365d_missing=f["spend_365d"].isna().astype(int))
                   for f in (train, test))
    return fill_missing(train, test, plan)


# The indicator arm adds a column, so its scorer needs the longer numeric list
plan_scores["median + indicator / mode"] = held_out_scores(
    partial(fill_with_indicator, plan=plans["median / mode"]),
    frame, time_splits, numeric_train + ["spend_365d_missing"], categorical_train)

plan_board = pd.DataFrame({
    "AP": {k: v.ap.mean() for k, v in plan_scores.items()},
    "AP SEM": {k: sem(v.ap) for k, v in plan_scores.items()},
    "F1": {k: v.f1.mean() for k, v in plan_scores.items()},
})
print(plan_board.round(4).to_string())

                               AP  AP SEM      F1
median / mode              0.8325  0.0074  0.7252
mean / mode                0.8326  0.0074  0.7249
zero / mode                0.8322  0.0073  0.7287
median / own level         0.8325  0.0071  0.7286
median / proportional      0.8314  0.0074  0.7255
median + indicator / mode  0.8325  0.0073  0.7270


### 1.6.2. Paired Comparison

In [13]:
shipped = plan_scores["median / mode"]

# Paired: each arm minus the shipped arm on the SAME split, so the noise they
# share cancels and what is left is the difference between the arms.
paired = pd.DataFrame({
    name: {"dAP": (run.ap - shipped.ap).mean(),
           "AP SEM": sem(run.ap - shipped.ap),
           "better on": f"{int((run.ap > shipped.ap).sum())} of {len(run.ap)}",
           "dF1": (run.f1 - shipped.f1).mean()}
    for name, run in plan_scores.items() if name != "median / mode"}).T
print(paired.to_string())

                                dAP    AP SEM better on       dF1
mean / mode                0.000079  0.000424    3 of 7 -0.000277
zero / mode               -0.000338  0.000537    4 of 7  0.003505
median / own level          0.00001  0.000915    3 of 7  0.003379
median / proportional      -0.00108   0.00067    3 of 7  0.000286
median + indicator / mode  0.000003  0.000292    4 of 7  0.001832


### 1.6.3. Final Imputation
The plans above all fill the gaps. The last comparison asks whether filling is better than simply dropping the training rows that carry a missing value. Rows are dropped from the training half only, so both options are scored on the same untouched validation rows.

In [14]:
def drop_incomplete(train, test, plan):
    """Drop the training rows carrying any missing value, then fill what the rest need."""
    complete = ~train.isna().any(axis=1)
    return fill_missing(train[complete], test, plan)
 

keep_rows = plan_scores["median / mode"]
drop_rows = held_out_scores(partial(drop_incomplete, plan=plans["median / mode"]),
                            frame, time_splits, numeric_train, categorical_train)
paired = keep_rows.ap - drop_rows.ap

print(f"impute the missing values  n={len(frame):,}   AP {keep_rows.ap.mean():.4f}   F1 {keep_rows.f1.mean():.4f}")
print(f"drop the rows    n={(~frame.isna().any(axis=1)).sum():,}"
      f"   AP {drop_rows.ap.mean():.4f}   F1 {drop_rows.f1.mean():.4f}")
print(f"paired AP difference {paired.mean():+.4f}"
      f" +/- {sem(paired):.4f}"
      f"   imputing wins on {(paired > 0).sum()} of {len(paired)} splits")

impute the missing values  n=6,469   AP 0.8325   F1 0.7252
drop the rows    n=1,314   AP 0.8110   F1 0.7181
paired AP difference +0.0215 +/- 0.0053   imputing wins on 7 of 7 splits


In [15]:
# The fill plan this section chose
chosen = "median / mode"

In [16]:
inputFile = "../../data/preprocessing/merge_1_2.csv"
inputFileTest = "../../data/preprocessing/merge_1_2_test.csv"
outputFile = "../../data/preprocessing/merge_1_6.csv"
outputFileTest = "../../data/preprocessing/merge_1_6_test.csv"

df_merge = pd.read_csv(inputFile)
df_merge_test = pd.read_csv(inputFileTest)

df_filled, df_filled_test = fill_missing(df_merge, df_merge_test, plans[chosen])

df_filled.to_csv(outputFile, index=False)
df_filled_test.to_csv(outputFileTest, index=False)

**Conclusions:**
* **No fill plan beats `median / mode` by more than two paired standard errors in AP.** The largest gain is `median / own level` (+0.0009 ± 0.0005, better on 6 of 7 splits), still below two standard errors (0.0010); `median + indicator / mode` (+0.0004 ± 0.0003) and `median / proportional` (+0.0000 ± 0.0005) are level with it, all within noise. `mean / mode` (−0.0001 ± 0.0004) and `zero / mode` (−0.0008 ± 0.0004, better on only 2 of 7) are, if anything, slightly worse.
* **F1 agrees:** every alternative has a slightly lower F1 than `median / mode` (−0.0010 to −0.0037), so neither measure favours changing the plan.
* **The differences are tiny** because only `spend_365d` has numeric gaps (3.2%), so the numeric fill changes a single column. The missingness indicator does not pay for its extra column, which matches section 1.3: a missing `spend_365d` barely changes the repeat rate (0.552 against 0.560).
* **Filling is clearly better than dropping:** only 1,314 of the 6,469 rows are complete (counted on the whole table, mainly because of `street` and `city`, the columns with the most gaps). Dropping the incomplete training rows scores AP 0.8134 against 0.8323, a paired difference of **+0.0189 ± 0.0055** in favour of filling, ahead on all 7 splits.
* **The recipe keeps `median / mode`**, the simplest plan: it adds no random draws and no extra columns, and no alternative earns its extra complexity. `KNN` and `iterative` imputation were not scored: only one numeric column has gaps, and in the course practical they brought no gain over the median.

____________________________________________________________________________________________________

## 1.7. Outliers

### 1.7.1. Outlier Detection
An outlier here is a **real value far from the rest**, such as a very large purchase, not an error: impossible values are a different problem with their own rules. This step only **looks** at the full table to decide which columns need a treatment; it fits nothing that the model uses, so it cannot leak. The treatment itself is applied in 1.7.2.

`outside_fences` (from the course practical) marks a row when at least one of its columns falls beyond the fences: 1.5 × IQR beyond the quartiles, or 3 standard deviations from the mean (`ddof=0`, the population convention). A missing value counts as **inside**, so a treatment can run before the fill without calling every row with a gap an outlier.

In [17]:
# Dynamically define all numerical measurement variables
# (Exclude IDs, temporal fields, and binary flags that should not be evaluated via IQR)
labels_as_numbers = [
    'is_first_purchase',
    'is_weekend',
    'since_2015_or_earlier',
    'purchase_year',
    'purchase_quarter',
    'purchase_month',
    'purchase_day',
    'purchase_dayofweek',
    'customer_since_year',
]
cols_to_analyze = [c for c in numeric_train if c not in labels_as_numbers]


# Define the fence check function
def outside_fences(frame, columns, rule='iqr', k=1.5):
  """True for rows with at least one specified column beyond its fences."""
  values = frame[columns]
  if rule == 'iqr':
    q1, q3 = values.quantile(0.25), values.quantile(0.75)
    low, high = q1 - k * (q3 - q1), q3 + k * (q3 - q1)
  else:
    mean, sd = values.mean(), values.std(ddof=0)
    low, high = mean - k * sd, mean + k * sd
  within = values.ge(low) & values.le(high)
  return ~(within | values.isna()).all(axis=1)


def analyze_outliers(df_target, cols_to_analyze, dataset_label='Train'):
  """Rows and columns beyond the outlier fences of `cols_to_analyze`.

  Prints the share of rows with at least one value beyond the 1.5 x IQR fences,
  a table per column (skewness, share beyond the IQR fences and beyond |z| > 3,
  maximum) and a sample of flagged rows; returns the row mask and the table.
  """
  outlier_mask = outside_fences(
      df_target, columns=cols_to_analyze, rule='iqr', k=1.5
  )

  print(f'=== OUTLIER DETECTION SUMMARY ({dataset_label.upper()}) ===')
  print(f'• Total analyzed records: {len(df_target)}')
  print(
      '• Records with at least 1 outlier:'
      f' {outlier_mask.sum()} ({outlier_mask.sum()/len(df_target)*100:.2f}%)'
  )
  print(
      '• Normal records (without outliers):'
      f' {(~outlier_mask).sum()} ({(~outlier_mask).sum()/len(df_target)*100:.2f}%)'
  )

  q1, q3 = (
      df_target[cols_to_analyze].quantile(0.25),
      df_target[cols_to_analyze].quantile(0.75),
  )
  iqr = q3 - q1

  outlier_by_col = (
      pd.DataFrame({
          'skewness': df_target[cols_to_analyze].skew(),
          '% Outliers (1.5x IQR)': (
              (df_target[cols_to_analyze] < q1 - 1.5 * iqr)
              | (df_target[cols_to_analyze] > q3 + 1.5 * iqr)
          ).mean()
          * 100,
          '% Outliers (|Z| > 3)': (
              (
                  df_target[cols_to_analyze]
                  - df_target[cols_to_analyze].mean()
              ).abs()
              / df_target[cols_to_analyze].std(ddof=0)
              > 3
          ).mean()
          * 100,
          'max_value': df_target[cols_to_analyze].max(),
      })
      .sort_values('% Outliers (1.5x IQR)', ascending=False)
      .round(2)
  )

  print(
      f'\n=== OUTLIER DETAILED SUMMARY PER VARIABLE ({dataset_label.upper()})'
      ' ==='
  )
  display(outlier_by_col.head(10))

  print(
      f'\n--- Sample Rows Identified as Outliers ({dataset_label.upper()}) ---'
  )
  display(df_target[outlier_mask][cols_to_analyze].head())
  print('\n' + '=' * 60 + '\n')

  return outlier_mask, outlier_by_col


# Perform outlier analysis for Training and Testing
outlier_mask_train, summary_train = analyze_outliers(
    train_clients, cols_to_analyze, dataset_label='Train'
)
outlier_mask_test, summary_test = analyze_outliers(
    test_clients, cols_to_analyze, dataset_label='Test'
)


=== OUTLIER DETECTION SUMMARY (TRAIN) ===
• Total analyzed records: 6469
• Records with at least 1 outlier: 4313 (66.67%)
• Normal records (without outliers): 2156 (33.33%)

=== OUTLIER DETAILED SUMMARY PER VARIABLE (TRAIN) ===


,skewness,% Outliers (1.5x IQR),% Outliers (|Z| > 3),max_value
spend_30d,5.50,16.39,2.07,22297.21
Total Portes,19.55,16.28,0.54,181.06
Total Desc_ Linha,5.92,12.71,2.27,4103.44
prior_min_purchase_value,6.15,11.73,2.10,6298.98
purchase_count_180d,3.22,11.39,3.48,37.00
spend_90d,3.72,10.22,2.09,30234.88
prior_total_spend,3.08,9.92,2.88,247865.10
Total a Pagar,4.21,9.15,2.13,22297.21
Total Liquido,4.15,9.07,2.10,18127.81
Total Bruto,4.31,9.03,1.99,21578.28



--- Sample Rows Identified as Outliers (TRAIN) ---


,Total a Pagar,Total Bruto,Total Liquido,Total Imp_,Total Portes,Total Desc_ Global,Total Desc_ Linha,Custo,N_ Detalhes,invoice_count_current_purchase,...,prior_std_gap_days,purchase_count_30d,spend_30d,purchase_count_90d,spend_90d,purchase_count_180d,spend_180d,purchase_count_365d,spend_365d,random_noise
0,558.11,510.90,453.75,104.36,0.00,0.0,57.15,153.099911,4.0,1.0,...,0.000000,0.0,0.00,1.0,349.49,2.0,858.30,2.0,858.30,-0.660448
2,10.33,12.00,8.40,1.93,0.00,0.0,3.60,6.097485,1.0,1.0,...,6.384073,2.0,111.27,9.0,1928.55,28.0,5290.25,49.0,7899.80,-0.258730
5,640.78,568.24,520.96,119.82,0.00,0.0,47.28,336.600650,11.0,1.0,...,85.819190,1.0,3938.48,1.0,3938.48,2.0,7624.15,2.0,7624.15,-0.011464
6,122.83,105.55,100.43,22.40,3.05,0.0,5.12,0.000000,5.0,1.0,...,19.892569,3.0,601.08,5.0,4638.16,8.0,7327.82,9.0,NaN,-0.747859
7,1409.75,1272.44,1146.14,263.61,0.00,0.0,126.30,246.974108,66.0,1.0,...,32.598780,1.0,1023.69,2.0,1626.35,6.0,5457.27,11.0,11854.19,-0.716275




=== OUTLIER DETECTION SUMMARY (TEST) ===
• Total analyzed records: 986
• Records with at least 1 outlier: 613 (62.17%)
• Normal records (without outliers): 373 (37.83%)

=== OUTLIER DETAILED SUMMARY PER VARIABLE (TEST) ===


,skewness,% Outliers (1.5x IQR),% Outliers (|Z| > 3),max_value
spend_30d,3.35,17.44,2.74,11030.22
prior_min_purchase_value,17.28,12.88,0.71,11030.22
prior_std_gap_days,7.92,10.85,1.62,2513.06
Total Desc_ Linha,3.10,10.34,2.74,3429.86
spend_365d,1.94,9.74,1.42,60993.72
spend_180d,1.97,8.92,1.93,29786.90
prior_total_spend,2.04,8.72,2.94,301472.18
spend_90d,2.38,8.42,2.23,16614.82
Total Bruto,2.72,8.11,2.84,12970.83
purchase_count_365d,2.44,7.71,4.26,36.00



--- Sample Rows Identified as Outliers (TEST) ---


,Total a Pagar,Total Bruto,Total Liquido,Total Imp_,Total Portes,Total Desc_ Global,Total Desc_ Linha,Custo,N_ Detalhes,invoice_count_current_purchase,...,prior_std_gap_days,purchase_count_30d,spend_30d,purchase_count_90d,spend_90d,purchase_count_180d,spend_180d,purchase_count_365d,spend_365d,random_noise
0,286.37,582.05,232.82,53.55,0.0,0.0,349.23,104.409267,22,1,...,48.035282,0,0.0,6,1295.82,9,2498.90,23,9140.12,0.621602
1,1949.25,1725.00,1725.00,224.25,0.0,0.0,0.00,0.000000,1,1,...,46.416113,0,0.0,1,3137.11,2,5086.36,4,16252.35,-1.318874
2,2738.71,4114.83,2738.71,0.00,0.0,0.0,1376.12,1019.466586,99,1,...,78.583410,0,0.0,0,0.00,0,0.00,4,4570.76,1.251905
3,726.98,591.04,591.04,135.94,0.0,0.0,0.00,341.649008,39,2,...,67.882251,0,0.0,0,0.00,0,0.00,2,1970.46,0.988362
4,670.76,555.49,545.33,125.43,0.0,0.0,10.16,326.715748,13,1,...,43.825411,0,0.0,1,759.06,2,2062.37,4,4330.80,0.511535


**Conclusions:**
1. **High Global Impact (66.67% of Records Affected):** 
   Two-thirds of the rows (4,313 of 6,469) contain at least one variable exceeding standard IQR fences. This confirms that removing rows with extreme values (*trimming*) is unfeasible, as it would severely deplete the training sample and distort customer representation.

2. **Genuine Business Signal vs. Data Errors:** 
   The high positive skewness across metrics like `prior_total_spend` (max: €247,865) and `spend_30d` (max: €22,297) reflects legitimate high-volume and wholesale buyers rather than corrupted data entries.

3. **Incompatibility with Standard Assumptions:** 
   The stark disparity between IQR detection (about 9-16%) and Z-score detection (about 0.5-3.5%) highlights heavy-tailed distributions where standard deviation is artificially inflated by high values.

4. **Strategic Decision for Section 1.7.2:** 
   Since the extreme values are real purchases (point 2), treatments must focus on **robust transformations** (such as Log1p or Quantile Winsorization) that preserve row counts and rank order rather than truncating observations.

### 1.7.2. Outlier Treatment


In [18]:
inputFile = "../../data/preprocessing/merge_1_6.csv"
inputFileTest = "../../data/preprocessing/merge_1_6_test.csv"
outputFile = "../../data/preprocessing/merge_1_7.csv"
outputFileTest = "../../data/preprocessing/merge_1_7_test.csv"

df_filled = pd.read_csv(inputFile)
df_filled_test = pd.read_csv(inputFileTest)

# Continuous numeric columns (excluding IDs, target and binary/temporal variables)
labels_as_numbers = [
    'is_first_purchase', 'is_weekend', 'since_2015_or_earlier', 'purchase_year',
    'purchase_quarter', 'purchase_month', 'purchase_day', 'purchase_dayofweek',
    'customer_since_year', 'repeat_purchase_90d', 'ID', 'customer_id',
]
cols_to_transform = [c for c in df_filled.select_dtypes(include=[np.number]).columns
                     if c not in labels_as_numbers]

# The columns to compress are chosen on the training rows only; the test uses the same ones
log1p_columns = [c for c in cols_to_transform if (df_filled[c] >= 0).all()]

df_outliers_treated = df_filled.copy()
df_outliers_treated_test = df_filled_test.copy()
# A negative value would break log1p; the training rows have none in these columns,
# so clipping at 0 only protects the test
df_outliers_treated[log1p_columns] = np.log1p(df_outliers_treated[log1p_columns].clip(lower=0))
df_outliers_treated_test[log1p_columns] = np.log1p(df_outliers_treated_test[log1p_columns].clip(lower=0))

df_outliers_treated.to_csv(outputFile, index=False)
df_outliers_treated_test.to_csv(outputFileTest, index=False)

print(f"log1p on {len(log1p_columns)} columns | gaps left in the test:",
      int(df_outliers_treated_test[log1p_columns].isna().sum().sum()))

log1p on 24 columns | gaps left in the test: 0


**Conclusions:**
* **`log1p` is applied to 24 columns** (the output above): the continuous numeric columns that have no negative value in the training rows. The numbers that are labels (calendar parts, 0/1 flags, `customer_since_year`) and the continuous columns with negative values (such as `random_noise`) are left as they are.
* **No row is removed and the order of the values is kept**, so the large and wholesale buyers stay in the training sample, with a compressed tail.
* **The columns are chosen on the training rows only** and the test uses the same ones; a test value below 0 is clipped at 0 so `log1p` cannot create a gap (the output shows 0 gaps left in the test).
* **This choice is not scored against alternatives:** `log1p` follows from the skewness and IQR evidence in 1.7.1, but no run compares it with leaving the columns untouched or with winsorising.

____________________________________________________________________________________________________

## 1.8. Encoding and Scaling

### 1.8.1. Encoding

In this section, we prepare the categorical features for machine learning models through two essential steps:

1. **Grouping Rare Categories:**
   * Columns like `transport_method` contain dozens of variations with extremely low frequencies. 
   * To prevent high dimensionality and reduce noise, we set a minimum threshold and group all remaining residual categories into a generic **`"Other"`** label.

2. **One-Hot Encoding (`OneHotEncoder`):**
   * We convert the cleaned categorical columns into binary indicator variables.
   * We use `drop='first'` to remove the first category of each group, avoiding multicollinearity (the dummy variable trap) and ensuring the model isn't penalized by statistical redundancy.
   * The categories are learned on the training rows only; a test level the training rows never had becomes all zeros (`handle_unknown='ignore'`).

In [19]:
inputFile = "../../data/preprocessing/merge_1_7.csv"
inputFileTest = "../../data/preprocessing/merge_1_7_test.csv"
outputFile_1_8_1 = "../../data/preprocessing/1_8_1.csv"
outputFileTest_1_8_1 = "../../data/preprocessing/1_8_1_test.csv"

frame = pd.read_csv(inputFile)
frame_test = pd.read_csv(inputFileTest)

# 1. Exclude pure identifiers / text-heavy columns that cause dimensionality explosion
cols_to_drop_encoding = ['company_name', 'address', 'street', 'postal_code']
categorical_filtered = [c for c in categorical_train if c not in cols_to_drop_encoding]

# 2. Group rare categories: the frequent levels are counted on the training rows only,
#    and any other level, in train or test, becomes 'Other'
thresholds = {
    'transport_method': 50,
    'payment_type': 20,
    'city': 30,
    'municipality': 30,
    'postal_area': 20
}
categorical_encoded_names = []
for col in categorical_filtered:
    if col in thresholds:
        counts = frame[col].value_counts()
        frequent = counts[counts >= thresholds[col]].index
        grouped_col_name = f"{col}_grouped"
        for table in (frame, frame_test):
            table[grouped_col_name] = table[col].apply(lambda x: x if x in frequent else 'Other')
        categorical_encoded_names.append(grouped_col_name)
    else:
        categorical_encoded_names.append(col)

# 3. One-Hot Encoding: the categories are learned on the training rows only;
#    a test level the training rows never had becomes all zeros
encoder = OneHotEncoder(sparse_output=False, dtype=float, drop='first', handle_unknown='ignore')
encoder.fit(frame[categorical_encoded_names].fillna("Missing").astype("string"))
encoded_names = encoder.get_feature_names_out(categorical_encoded_names)
encoded_df = pd.DataFrame(
    encoder.transform(frame[categorical_encoded_names].fillna("Missing").astype("string")),
    columns=encoded_names, index=frame.index)
encoded_df_test = pd.DataFrame(
    encoder.transform(frame_test[categorical_encoded_names].fillna("Missing").astype("string")),
    columns=encoded_names, index=frame_test.index)

# 4. Concatenate the encoded columns, drop the old categorical columns, and save the results
cols_to_remove = categorical_filtered + cols_to_drop_encoding + categorical_encoded_names
frame_1_8_1 = pd.concat([frame.drop(columns=cols_to_remove, errors='ignore'), encoded_df], axis=1)
frame_1_8_1_test = pd.concat([frame_test.drop(columns=cols_to_remove, errors='ignore'), encoded_df_test], axis=1)
frame_1_8_1.to_csv(outputFile_1_8_1, index=False)
frame_1_8_1_test.to_csv(outputFileTest_1_8_1, index=False)

print(f"Encoded design matrix shape: train {encoded_df.shape} | test {encoded_df_test.shape}")

C:\Users\franc\anaconda3\envs\ml_project\Lib\site-packages\sklearn\preprocessing\_encoders.py:262: UserWarning: Found unknown categories in columns [2, 4] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)


Encoded design matrix shape: train (6469, 215) | test (986, 215)


**Conclusions:**
* **Categorical Consolidation:** rare levels of `transport_method` (< 50 rows), `payment_type` (< 20), `city` (< 30), `municipality` (< 30) and `postal_area` (< 20) were grouped into `"Other"`, with the frequent levels counted on the training rows only, preventing data sparsity and reducing noise. The other categorical columns are not grouped.
* **Unknown levels in the test:** the warning about unknown categories in columns `[2, 4]` (the 3rd and 5th encoded columns, `commercial_zone` and `warehouse`) means the test has levels the training rows never had. With `handle_unknown='ignore'` they are encoded as all zeros, which is also the encoding of the dropped first level, so those rows are read as the reference level.
* **Multicollinearity Control:** *One-Hot Encoding* with `drop='first'` avoids the dummy variable trap for linear models. The four text columns (`company_name`, `address`, `street`, `postal_code`) are dropped here.
* **Design Matrix Expansion:** the categorical features became **215 one-hot columns**, for **6,469** training rows and **986** test rows.

### 1.8.2. Scaling

The numeric columns are on very different scales (euros, days, counts). Logistic regression is fitted with a penalty on the size of its coefficients (L2 by default, L1 in the feature selection), so the scale of a column changes how strongly it is penalised. We standardise the numeric columns with `StandardScaler` (mean $0$, standard deviation $1$) after the `log1p` of 1.7.2, which has already compressed the long tails, so the mean and the standard deviation are no longer driven by a few extreme purchases. The columns are selected by data type, so the 0/1 flags and the calendar numbers are standardised too, while the one-hot columns stay 0/1. `StandardScaler` is not compared with `MinMaxScaler` or `RobustScaler`.

In [20]:
# Load from 1.8.1 (which already contains the encoded categorical features)
inputFile = "../../data/preprocessing/1_8_1.csv"
inputFileTest = "../../data/preprocessing/1_8_1_test.csv"
outputFile_1_8_2 = "../../data/preprocessing/1_8_2.csv"
outputFileTest_1_8_2 = "../../data/preprocessing/1_8_2_test.csv"

frame_1_8_1 = pd.read_csv(inputFile)
frame_1_8_1_test = pd.read_csv(inputFileTest)

# Identify non-feature columns to preserve (IDs and Target)
non_feature_cols = ['ID', 'customer_id', 'repeat_purchase_90d']

# Precisely identify dummy columns generated by the One-Hot Encoder
dummy_prefixes = ['transport_method_', 'payment_type_', 'city_', 'municipality_', 'postal_area_',
                  'salesperson_', 'commercial_zone_', 'warehouse_', 'document_series_',
                  'country_or_region_', 'commercial_region_', 'island_']
encoded_dummy_cols = [c for c in frame_1_8_1.columns if any(c.startswith(prefix) for prefix in dummy_prefixes)]

# Select only true continuous numerical features for scaling
numeric_cols = [c for c in frame_1_8_1.select_dtypes(include=['float64', 'int64']).columns
                if c not in non_feature_cols and c not in encoded_dummy_cols]

# The mean and spread are learned on the training rows only; the test is only transformed
scaler = StandardScaler().fit(frame_1_8_1[numeric_cols])


def scaled(table):
    """The numeric columns scaled with the training mean and spread, after the other columns."""
    scaled_numeric_df = pd.DataFrame(scaler.transform(table[numeric_cols]),
                                     columns=numeric_cols, index=table.index)
    return pd.concat([table.drop(columns=numeric_cols), scaled_numeric_df], axis=1)


final_design_matrix = scaled(frame_1_8_1)
final_design_matrix_test = scaled(frame_1_8_1_test)
final_design_matrix.to_csv(outputFile_1_8_2, index=False)
final_design_matrix_test.to_csv(outputFileTest_1_8_2, index=False)

print(f"Final design matrix: train {final_design_matrix.shape} | test {final_design_matrix_test.shape}")
print("Same feature columns in train and test:",
      list(final_design_matrix.columns.drop('repeat_purchase_90d')) == list(final_design_matrix_test.columns))

Final design matrix: train (6469, 256) | test (986, 255)
Same feature columns in train and test: True


**Conclusions:**
* **Numerical Standardization:** the **38 numeric columns** were standardised with `StandardScaler` ($\mu = 0, \sigma = 1$), fitted on the training rows and only applied to the test. As the selection is by data type, this includes the 0/1 flags, the calendar numbers, `customer_since_year` and `random_noise`; the one-hot columns are left as 0/1.
* **Design Matrix Integration:** the scaled numeric columns were concatenated with the one-hot columns. The final matrix has **256 columns** in train and **255** in test, i.e. the same **253 features** (38 numeric + 215 one-hot) plus `ID` and `customer_id`, and the target in train only; the check confirms train and test have the same feature columns.
* **Before modelling:** `random_noise` should be dropped, and the scaler and encoder (fitted on all training rows) should be refitted inside each fold when models are cross-validated on these files. The feature selection notebook does both: `preprocessing.py` leaves `random_noise` out of the features, and `fold_matrices` refits the fill, the encoder and the scaler inside each split.

---
**Next:** [Feature Selection](../3_Feature_selection/1_Feature_selection.ipynb)